# 한국어 의존구문 feature 통제 비교

주어·목적어·수식어·절·접속·서술어의 개수와 토큰 대비 비율 12개를 기존 문자 TF-IDF에 추가합니다. 영어 논문의 feature를 그대로 복제하지 않고 한국어 UD 관계에 대응되는 최소 블록만 사용합니다.

재현 명령: `python -m scripts.evaluation.dependency_features --model-dir E:\\rfp-models\\stanza`

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib import font_manager

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'scripts').is_dir()), Path.cwd().resolve())
installed = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next((f for f in ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'DejaVu Sans'] if f in installed), 'DejaVu Sans')
plt.rcParams['axes.unicode_minus'] = False
dependency = json.loads((ROOT / 'data/processed/dependency_results.json').read_text(encoding='utf-8'))
scores = pd.DataFrame({'기준선': {k: v['fold_mean'] for k, v in dependency['baseline']['metrics'].items()}, 'TF-IDF + 의존구문': {k: v['fold_mean'] for k, v in dependency['variant']['metrics'].items()}}).T
display(scores[['macro_f1', 'accuracy', 'review_precision', 'review_recall', 'review_f1']].style.format('{:.3f}').highlight_max(axis=0, color='#b7e4c7'))
folds = pd.DataFrame(dependency['comparison']['per_fold'], columns=['평가 문서', '기준선', '의존구문'])
folds['차이'] = folds['의존구문'] - folds['기준선']
importance = pd.Series(dependency['coefficients']['mean_absolute']).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(folds['평가 문서'], folds['차이'], color=['#2a9d8f' if value > 0 else '#e76f51' for value in folds['차이']])
axes[0].axvline(0, color='black', linewidth=1); axes[0].set_title('의존구문 추가 시 문서별 macro F1 변화')
importance.plot.barh(ax=axes[1], color='#457b9d'); axes[1].set_title('의존구문 feature 평균 |계수|')
plt.tight_layout(); plt.show()
eta = dependency['distribution_eta_squared']
print(f"평균 차이 {dependency['comparison']['mean_delta']:+.3f}, 우세 {dependency['comparison']['wins']}/10")
print(f"의존구문 feature 평균 η²: 문서 {eta['document']['mean']:.3f} / 라벨 {eta['label']['mean']:.3f}")